# 1 · Meet the Beast

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=01-meet-the-beast.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/01-meet-the-beast.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** on
the site to switch story / how-to / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — your companion creature
:class: storytelling

*You are an adventurer, and every adventurer keeps a companion creature. You were
entrusted with **the Beast** — powerful, but not yet yours to command. Today the
masters simply **introduce** you: they show, on the Beast itself, what it can do.
They warm it from within, then grab a metal plate and stretch it to show how
robustly the Beast handles even large deformations. You only watch — the real
training starts next unit.*
:::

Almost every NGSolve session is the **same three steps**, no matter how hard the
problem: **(1)** conjure a **geometry and mesh**, **(2)** **solve a variational
problem** on it, **(3)** **visualize** the result. We will walk that loop **twice** —
once for a linear **Poisson** problem and once for a **nonlinear elasticity** problem —
so you see the shape of everything to come. Features fly past (boundary conditions,
solvers, nonlinear iterations); we name them and move on. Each gets its own unit later.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve import solvers
from ngsolve.webgui import Draw

## 1. Geometry & mesh — the Beast

The Beast is NGSolve's logo **sculpture**: a thick spherical shell pierced by three
orthogonal cylindrical bores. We build it with **Netgen/OCC** constructive geometry
(sphere and cylinder primitives combined with boolean `-`), then let Netgen mesh it.
Naming a few faces now lets us attach boundary conditions later.

In [ ]:
def beast_sculpture():
    s = Sphere(Pnt(50, 50, 50), 80) - Sphere(Pnt(50, 50, 50), 50)
    for p, d in [(Pnt(-100, 0, 0), X), (Pnt(100, -100, 100), Y), (Pnt(0, 100, -100), Z)]:
        s = s - Cylinder(p, d, r=40, h=300)
    return s.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)     # centre + shrink

beast = beast_sculpture()
mesh = Mesh(OCCGeometry(beast).GenerateMesh(maxh=0.7))
mesh.Curve(2)
print(f"the Beast: {mesh.nv} vertices, {mesh.ne} elements")
Draw(mesh)

## 2. A variational solve — fire-energy inside the Beast

Legend says the Beast stores the energy for its fire-breath deep in its body. We model
that as a **heat source** living inside the shell and ask for the steady temperature,
the **Poisson problem** $-\Delta u = f$ with $u=0$ on the surface. In NGSolve this is a
**weak form** on an `H1` space; we assemble it and solve with the **modern solver
interface** — a `Preconditioner` plus a conjugate-gradient iteration (`solvers.CG`).
(Solvers get their own unit, 6; richer ones — multigrid, and add-ons like
[ngsPETSc](https://ngsolve.org) — live there.)

In [ ]:
r = sqrt(x*x + y*y + z*z)
source = 40 * exp(-((r - 3.25) / 0.5)**2)            # a glow buried in the shell

fes = H1(mesh, order=2, dirichlet=".*")              # u = 0 on the whole surface
u, v = fes.TnT()
a = BilinearForm(grad(u) * grad(v) * dx)
pre = Preconditioner(a, "local")                     # build the preconditioner with the form
a.Assemble()
f = LinearForm(source * v * dx).Assemble()

gfu = GridFunction(fes)
solvers.CG(mat=a.mat, rhs=f.vec, sol=gfu.vec, pre=pre.mat, tol=1e-8,
           maxsteps=1000, printrates=False)
print(f"hottest point inside the Beast: {max(gfu.vec):.2f}")
Draw(gfu, mesh, "temperature")

## 3. The same loop, nonlinear — stretching a Swiss-cross plate

To show off **robustness**, the masters grab a stiff specimen — a raised **Swiss-flag
block**, a box with the cross punched out — clamp one end and **pull** the other. Large
deformation means **geometric nonlinearity**: we use a **hyperelastic (Neo-Hookean)**
stored energy $\psi(F)$ of the deformation gradient $F=I+\nabla u$, and let NGSolve find
the displacement that **minimises the total energy**. That is a nonlinear system, solved
by **Newton's method** (`solvers.Newton`), ramping the pulling traction in a few steps so
each Newton solve stays in its basin. The same *geometry → weak form → solve → draw* loop —
only the weak form is now an energy, and the solve iterates.

In [ ]:
W, t, bw, bl = 6.0, 0.6, 1.2, 4.0                    # plate size, thickness, cross width/length
plate = Box(Pnt(0, 0, 0), Pnt(W, W, t))
lo, hi = (W - bl) / 2, (W + bl) / 2
clo, chi = W / 2 - bw / 2, W / 2 + bw / 2
flag = (plate - Box(Pnt(clo, lo, -0.1), Pnt(chi, hi, t + 0.1))   # vertical cross bar
              - Box(Pnt(lo, clo, -0.1), Pnt(hi, chi, t + 0.1)))  # horizontal cross bar
flag.faces.Min(X).name = "hold"                      # clamped end
flag.faces.Max(X).name = "pull"                      # the masters pull here
fmesh = Mesh(OCCGeometry(flag).GenerateMesh(maxh=0.6)); fmesh.Curve(1)

E, nu = 200.0, 0.35                                  # Young's modulus, Poisson ratio
mu, lam = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))
V = VectorH1(fmesh, order=1, dirichlet="hold")
ud = V.TrialFunction()
F = Id(3) + Grad(ud); J = Det(F); C = F.trans * F
psi = 0.5 * mu * (Trace(C) - 3) - mu * log(J) + 0.5 * lam * log(J)**2   # Neo-Hooke

traction = Parameter(0.0)
elastic = BilinearForm(V, symmetric=True)
elastic += Variation(psi * dx)                       # stored elastic energy
elastic += Variation(-traction * ud[0] * ds("pull"))  # work of the pulling traction

gfd = GridFunction(V); gfd.vec[:] = 0
for fac in [0.2, 0.4, 0.6, 0.8, 1.0]:                # ramp the load, keep continuation
    traction.Set(22.0 * fac)
    solvers.Newton(elastic, gfd, inverse="sparsecholesky", dampfactor=0.5, printing=False)
print(f"the plate stretched by {max(abs(gfd.vec.FV().NumPy())):.2f} units — and held")

We draw it as a **morph**: a multidim field whose first frame is the **original block**
and whose second frame is the **fully pulled** shape. In the webgui, drag the
**multidim** slider (or press play) to switch between the undeformed and deformed states
— the surface warps with the displacement and is coloured by its magnitude.

In [ ]:
morph = GridFunction(V)
morph.vec[:] = 0                                     # frame 0: the original, undeformed block
morph.AddMultiDimComponent(gfd.vec)                  # frame 1: the fully pulled shape
Draw(morph, fmesh, "displacement", deformation=True,
     interpolate_multidim=True, animate=True)

:::{dropdown} 📚 Further reading
:class: further-reading

- **The Poisson solve, step by step** — i-tutorial
  [1.1 First NGSolve example](https://docu.ngsolve.org/latest/i-tutorials/unit-1.1-poisson/poisson.html).
- **Hyperelasticity & Newton** — i-tutorial
  [3D Solid Mechanics](https://docu.ngsolve.org/latest/i-tutorials/wta/elasticity3D.html)
  and the Newton loop in
  [3.3 Nonlinear problems](https://docu.ngsolve.org/latest/i-tutorials/unit-3.3-nonlinear/nonlinear.html);
  a fuller derivation in the
  [SciCADE course — 3D elasticity](https://jschoeberl.github.io/SciCADE-course/unit2-elasticity/elasticity3D.html).
:::

:::{dropdown} 🧠 Quiz — what were the two "solves" really doing?
:class: quiz

Both followed the identical loop **geometry → weak form → solve → draw**. The
**difference** is the weak form: Poisson is **linear** (one matrix solve — here an
iterative `CG`), elasticity is **nonlinear** (an energy whose minimiser we chase with
**Newton**, each step itself a linear solve). Everything else you saw — `H1` vs
`VectorH1`, `dirichlet` boundary tags, a `Preconditioner`, `Variation`, load ramping —
are the building blocks the rest of Part I unpacks one at a time.
:::

**Next:** the masters withdraw. To approach the Beast yourself you first conjure it some
**food** — and learn elementary **geometry & meshing** along the way (unit 2).

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "02-geometry", "2 · Conjuring geometry & taming the mesh 🍫"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))